# Experimento 5 — SVM com SMOTE utilizando todas as variáveis

O quinto experimento do modelo SVM explora uma nova estratégia para lidar com o forte desbalanceamento das classes ambientais presentes no dataset AquaSense.

Nos experimentos anteriores, o balanceamento foi realizado através do parâmetro `class_weight="balanced"`, que altera matematicamente o peso dos erros cometidos nas classes minoritárias. Apesar de melhorar significativamente o recall da classe `Crítica`, essa abordagem ainda apresentava limitações importantes na separação entre algumas categorias ambientais.

Neste experimento, utilizaremos o algoritmo **SMOTE (Synthetic Minority Oversampling Technique)**.

O SMOTE funciona criando artificialmente novas amostras sintéticas das classes minoritárias a partir da interpolação entre vizinhos próximos. Dessa forma, o modelo passa a treinar em um conjunto de dados muito mais equilibrado, reduzindo o viés causado pela predominância da classe `Excelente`.

Diferentemente do Experimento 4:
- não utilizaremos `class_weight`;
- o balanceamento ocorrerá diretamente nos dados de treino.

Além disso, continuaremos utilizando o conjunto ampliado de variáveis ambientais introduzido no Experimento 3.

## Objetivo do Experimento

O objetivo principal é verificar se:
- o SVM consegue aumentar ainda mais o recall da classe `Crítica`;
- sem perder excessivamente a estabilidade geral do modelo;
- e mantendo desempenho competitivo em relação aos demais algoritmos do projeto.

## Variáveis utilizadas

As variáveis utilizadas foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

## Variável alvo

A variável alvo utilizada foi:

- `conama_status`

contendo as quatro classes ambientais:
- Excelente;
- Boa;
- Atenção;
- Crítica.

## Divisão dos dados

O dataset foi dividido em:
- 80% treino;
- 20% teste.

O SMOTE será aplicado exclusivamente no conjunto de treino para evitar vazamento de dados (*data leakage*).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path(
        "C:/Users/anaju\OneDrive/projetos_lucas/AquaSense-ML/amostra_rotulada.parquet"
    )

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente local/VS Code detectado.
Dataset carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


## Preparação do ambiente de Machine Learning

Nesta etapa serão importadas as bibliotecas responsáveis por:
- pré-processamento dos dados;
- divisão treino/teste;
- padronização das variáveis;
- construção do pipeline;
- aplicação do SMOTE;
- treinamento do SVM;
- cálculo das métricas de avaliação.

O pipeline do `imblearn` será utilizado para garantir que o SMOTE seja aplicado apenas nos dados de treino.

In [3]:
!pip install imbalanced-learn


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\anaju\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("Bibliotecas de ML carregadas.")

Bibliotecas de ML carregadas.


In [5]:
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

print("X e y definidos.")

X e y definidos.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [7]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Pré-processamento configurado.")

Pré-processamento configurado.


## Construção do Pipeline com SMOTE

Nesta etapa o pipeline será montado utilizando:
- pré-processamento;
- geração sintética de amostras via SMOTE;
- treinamento do modelo SVM.

O SMOTE será executado somente após a transformação dos dados e apenas dentro do conjunto de treino.

Isso evita vazamento de informação e garante validade estatística ao experimento.

In [8]:
from sklearn.svm import LinearSVC

model_smote = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "smote",
            SMOTE(random_state=SEED)
        ),

        (
            "classifier",
            LinearSVC(
                random_state=SEED,
                max_iter=5000
            )
        )
    ]
)

print("Treinando SVM com SMOTE...")

model_smote.fit(X_train, y_train)

print("Treinamento concluído.")

Treinando SVM com SMOTE...
Treinamento concluído.


## Avaliação das Métricas de Treino

In [9]:
y_train_pred = model_smote.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Train Accuracy:
0.7717005984847816

Train Precision:
0.801922654340973

Train Recall:
0.7717005984847816

Train F1:
0.7783541319815848

Train Confusion Matrix:
[[ 3629   943  2904    84]
 [ 3493  6850  3551  7784]
 [  295    95   713     3]
 [  212  4472  1989 76102]]


## Avaliação no Conjunto de Teste

In [10]:
y_pred = model_smote.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7732673267326733

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.50      0.47      0.48      1890
         Boa       0.56      0.32      0.41      5419
     Crítica       0.08      0.66      0.14       277
   Excelente       0.91      0.92      0.91     20694

    accuracy                           0.77     28280
   macro avg       0.51      0.59      0.49     28280
weighted avg       0.80      0.77      0.78     28280


Confusion Matrix:
[[  892   230   749    19]
 [  786  1742   918  1973]
 [   68    24   184     1]
 [   46  1115   483 19050]]


## Resultados Obtidos — Experimento 5

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.773
```

O Experimento 5 avaliou o uso do SMOTE no SVM com todas as variáveis ambientais. Diferente do Experimento 4, que utilizou `class_weight="balanced"`, este experimento tentou corrigir o desbalanceamento criando amostras sintéticas das classes minoritárias.



## Comparação entre algoritmos — Experimento 5

| Métrica | RF Exp. 5 | LightGBM Exp. 5 | Regressão Logística Exp. 5 | SVM Exp. 5 |
|---|---:|---:|---:|---:|
| Accuracy teste | — | — | 0.788 | 0.773 |
| Precision Atenção | — | — | 0.65 | 0.50 |
| Precision Boa | — | — | 0.54 | 0.56 |
| Precision Crítica | — | — | 0.11 | 0.08 |
| Precision Excelente | — | — | 0.91 | 0.91 |
| Recall Atenção | — | — | 0.43 | 0.47 |
| Recall Boa | — | — | 0.49 | 0.32 |
| Recall Crítica | — | — | 0.65 | 0.66 |
| Recall Excelente | — | — | 0.90 | 0.92 |
| F1 Atenção | — | — | 0.52 | 0.48 |
| F1 Boa | — | — | 0.52 | 0.41 |
| F1 Crítica | — | — | 0.19 | 0.14 |
| F1 Excelente | — | — | 0.91 | 0.91 |
| Macro avg F1 | — | — | 0.53 | 0.49 |
| Weighted avg F1 | — | — | 0.80 | 0.78 |



## Análise das métricas

O SVM com SMOTE apresentou accuracy de 0.773, resultado praticamente igual ao obtido no Experimento 4 com `class_weight="balanced"`.

A principal mudança ocorreu no comportamento das classes minoritárias. A classe `Crítica` atingiu recall de 0.66, superando levemente o Experimento 4, no qual o recall havia sido 0.62. Isso indica que o SMOTE ajudou o modelo a identificar mais amostras críticas.

Entretanto, essa melhora veio acompanhada de baixa precision para `Crítica`, que ficou em 0.08. Isso significa que o modelo passou a prever muitas amostras como críticas incorretamente, aumentando falsos positivos.

A classe `Atenção` apresentou recall de 0.47 e F1-score de 0.48. O resultado foi razoável, mas inferior ao desempenho da Regressão Logística com SMOTE nessa classe.

A classe `Boa` foi uma das mais prejudicadas no SVM com SMOTE. Apesar de precision de 0.56, o recall ficou em apenas 0.32, mostrando que muitas amostras reais dessa classe foram classificadas como `Excelente` ou `Crítica`.

A classe `Excelente` manteve desempenho alto, com recall de 0.92 e F1-score de 0.91. Isso demonstra que, mesmo com SMOTE, o modelo ainda preservou boa capacidade de reconhecer a classe majoritária.



## Análise da Matriz de Confusão

A matriz de confusão mostra que o modelo identificou corretamente:

- 892 amostras da classe `Atenção`;
- 1742 amostras da classe `Boa`;
- 184 amostras da classe `Crítica`;
- 19050 amostras da classe `Excelente`.

O ponto positivo foi a identificação de 184 das 277 amostras críticas, representando o melhor recall de `Crítica` obtido pelo SVM até o momento.

Por outro lado, o modelo classificou muitas amostras de outras classes como `Crítica`, especialmente:
- 749 amostras reais de `Atenção`;
- 918 amostras reais de `Boa`;
- 483 amostras reais de `Excelente`.

Isso explica a baixa precision da classe `Crítica`.


## Conclusão

O Experimento 5 demonstrou que o SMOTE aumentou a sensibilidade do SVM para a classe `Crítica`, alcançando recall de 0.66. Esse foi o melhor resultado do SVM até agora para essa classe.

Entretanto, o ganho de recall veio acompanhado de perda de precisão, especialmente na classe `Crítica`, indicando aumento significativo de falsos alarmes.

Comparando com a Regressão Logística com SMOTE, o SVM apresentou:
- menor accuracy;
- menor macro avg F1;
- menor weighted avg F1;
- recall semelhante para `Crítica`;
- pior desempenho nas classes `Atenção` e `Boa`.

Assim, o SMOTE mostrou-se útil para aumentar a detecção de casos críticos, mas não necessariamente melhorou o equilíbrio geral do SVM.

Entre os experimentos do SVM, o Experimento 5 é relevante por maximizar a detecção de `Crítica`, mas o Experimento 4 ainda parece mais equilibrado, pois apresentou desempenho semelhante com melhor distribuição geral das métricas.

Para os próximos experimentos, recomenda-se testar ajuste de hiperparâmetros ou estratégias híbridas, buscando manter o alto recall da classe `Crítica` sem aumentar excessivamente os falsos positivos.